In [13]:
import requests
from bs4 import BeautifulSoup


def get_ayam_recipes():
    url = "https://resepika.com/"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        # Target link elements specifically to extract both text and the URL
        links = soup.find_all("a")

        print("--- Recipes containing 'Ayam' ---")
        found_any = False
        seen_titles = set()

        for link_element in links:
            title_text = link_element.get_text(strip=True)
            href = link_element.get("href")

            # Check if "ayam" is in the title string (case-insensitive)
            if href and "ayam" in title_text.lower():
                # Clean up relative URLs to absolute URLs if necessary
                if href.startswith("/"):
                    full_url = f"https://resepika.com{href}"
                elif href.startswith("http"):
                    full_url = href
                else:
                    continue

                # Avoid printing duplicates or empty links
                if title_text not in seen_titles and len(title_text) > 3:
                    print(f"Title: {title_text}")
                    print(f"Link : {full_url}")
                    print("-" * 30)

                    seen_titles.add(title_text)
                    found_any = True

        if not found_any:
            print("No recipes found with 'ayam' on the homepage.")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching the website: {e}")


if __name__ == "__main__":
    get_ayam_recipes()

--- Recipes containing 'Ayam' ---
Title: belanja saya nasi ayam gepuk 🍗
Link : https://donate.stripe.com/00gcNN0oc7uId5SaEH
------------------------------
Title: Nasi Tomato & Ayam Masak Merah Kenduri
Link : https://www.instagram.com/p/DHsGA4xSh5g/
------------------------------
Title: Rendang Satu Ekor Ayam
Link : https://www.instagram.com/p/DHkhhsryIMs/
------------------------------
Title: Ayam Goreng Masak Lemak Cili Api
Link : https://www.instagram.com/p/DHc62H8yNbB/
------------------------------
Title: Ayam Percik & Sirap Bandung Soda
Link : https://www.instagram.com/p/DHXsFyASLXG/
------------------------------
Title: Ayam Goreng Sambal Kantan
Link : https://www.instagram.com/p/DHICMnDyrbm/
------------------------------
Title: Ayam Goreng Pandan cicah Sos Thai
Link : https://www.instagram.com/p/DHAT2qMS97K/
------------------------------
Title: Ayam Kukus Sambal Embe
Link : https://www.instagram.com/p/DG4kzaHyG_K/
------------------------------
Title: Sup Ikan dan Sup Ayam Sia

In [14]:
import json
import requests
from bs4 import BeautifulSoup


def get_ayam_recipes():
    url = "https://resepika.com/"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        links = soup.find_all("a")

        recipes_list = []
        seen_titles = set()

        for link_element in links:
            title_text = link_element.get_text(strip=True)
            href = link_element.get("href")

            # Ignore stripe donation links and matching text fragments
            if (
                not href
                or "donate.stripe.com" in href
                or "belanja saya" in title_text.lower()
            ):
                continue

            if "daging" in title_text.lower():
                if href.startswith("/"):
                    full_url = f"https://resepika.com{href}"
                elif href.startswith("http"):
                    full_url = href
                else:
                    continue

                if title_text not in seen_titles and len(title_text) > 3:
                    seen_titles.add(title_text)

                    # Generate id slug safely
                    recipe_id = (
                        title_text.lower()
                        .replace(" ", "-")
                        .replace(",", "")
                        .replace(".", "")
                        .strip("-")
                    )

                    recipe_data = {
                        "id": recipe_id,
                        "title": title_text,
                        "source": "khairul-aming",
                        "videoUrl": full_url,
                        "ingredients": [],  # Leaving empty for your upcoming method
                        "prepTime": "45 min",
                        "servings": 4,
                    }

                    recipes_list.append(recipe_data)
                    print(f"Added: {title_text}")

        # Write data file out cleanly
        output_data = {"recipes": recipes_list}
        with open("scraped_recipes.json", "w", encoding="utf-8") as f:
            json.dump(output_data, f, indent=2, ensure_ascii=False)

        print(f"\nSaved {len(recipes_list)} recipes to scraped_recipes.json!")

    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching the website: {e}")


if __name__ == "__main__":
    get_ayam_recipes()

Added: Daging Belengas
Added: Sup Daging Thai
Added: Rendang 2 Kilo Daging
Added: Puasa ke-24 : Daging Dendeng
Added: Puasa Ke-5 : Daging Harimau Menangis
Added: Kari Daging Raya Haji 🥩
Added: Puasa Hari 16 (Part 1) - Daging Goreng Kunyit
Added: Daging Masak Kicap
Added: Daging Harimau Menangis
Added: Daging Black Pepper 🥩
Added: Daging Balado
Added: Puasa Day 16 - Daging Bakar cicah Thai Sauce
Added: Puasa Day 18: Daging Black Pepper

Saved 13 recipes to scraped_recipes.json!


crawler

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

def crawl(url, max_pages=2000):
    visited = set()
    to_visit = [url]
    domain = urlparse(url).netloc

    while to_visit and len(visited) < max_pages:
        current_url = to_visit.pop(0)
        if current_url in visited:
            continue
            
        try:
            response = requests.get(current_url, timeout=5)
            visited.add(current_url)
            print(f"Found: {current_url}")
            
            soup = BeautifulSoup(response.text, 'html.parser')
            for link in soup.find_all('a', href=True):
                full_url = urljoin(current_url, link['href'])
                # Only follow links within the same website
                if urlparse(full_url).netloc == domain and full_url not in visited:
                    to_visit.append(full_url)
        except Exception as e:
            continue

crawl("https://khairulaming.my")


Found: https://khairulaming.my
Found: https://khairulaming.my/
Found: https://khairulaming.my/daging-dendeng/
Found: https://khairulaming.my/proses-mendapatkan-sijil-halal-untuk-dendeng-dan-sambal-nyet/
Found: https://khairulaming.my/pengiktirafan-halal-untuk-dendeng-dan-sambal-nyet/
Found: https://khairulaming.my/salam-dari-china-dengan-kejayaan-kempen-11-11/
Found: https://khairulaming.my/category/kenyataan/
Found: https://khairulaming.my/category/aktiviti/
Found: https://khairulaming.my/category/produk/
Found: https://khairulaming.my/category/resipi/
Found: https://khairulaming.my/category/umum/
Found: https://khairulaming.my/category/series/
Found: https://khairulaming.my/daging-dendeng/#content
Found: https://khairulaming.my/ais-kepal-milo-fruit-cocktail/
Found: https://khairulaming.my/cadbury-tart/
Found: https://khairulaming.my/kek-coklat-kukus/
Found: https://khairulaming.my/proses-mendapatkan-sijil-halal-untuk-dendeng-dan-sambal-nyet/#content
Found: https://khairulaming.my/ban

khairulaming.my scraper

In [15]:
import json
import os
from urllib.parse import urljoin, urlparse
import requests
from bs4 import BeautifulSoup


def load_valid_ingredient_ids():
    path = r"C:\GetSTACKED2024\masakapa\src\data\ingredients.json"
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
                if isinstance(data, list):
                    return [item["id"] for item in data if "id" in item]
                elif isinstance(data, dict):
                    return list(data.keys())
        except Exception as e:
            print(f"Gagal membaca ingredients.json: {e}")
    return []


def generate_recipe_id(title):
    return (
        title.lower()
        .replace(" ", "-")
        .replace(",", "")
        .replace(".", "")
        .strip("-")
    )


def crawl_khairul_aming_recipes(start_url, max_pages=50):
    visited = set()
    to_visit = [start_url]
    domain = urlparse(start_url).netloc

    recipes_list = []
    seen_recipe_ids = set()
    valid_ids = load_valid_ingredient_ids()

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    print("--- Starting Deep Scan on khairulaming.my ---")

    while to_visit and len(visited) < max_pages:
        current_url = to_visit.pop(0)
        if current_url in visited:
            continue

        try:
            response = requests.get(current_url, headers=headers, timeout=5)
            visited.add(current_url)

            if "?" in current_url or "/tag/" in current_url:
                continue

            soup = BeautifulSoup(response.text, "html.parser")

            h1_element = soup.find("h1")
            page_title = h1_element.get_text(strip=True) if h1_element else ""

            if "ayam" in current_url.lower() or "ayam" in page_title.lower():
                recipe_id = generate_recipe_id(
                    page_title
                    if page_title
                    else urlparse(current_url).path.strip("/")
                )

                if recipe_id and recipe_id not in seen_recipe_ids:
                    print(f"\nTarget Match Found! Processing page: {current_url}")

                    containers = soup.find_all(
                        class_="elementor-widget-container"
                    )
                    combined_text = ""
                    for container in containers:
                        combined_text += (
                            " "
                            + container.get_text(
                                separator=" ", strip=True
                            ).lower()
                        )

                    matched_ingredients = []
                    for ingredient_id in valid_ids:
                        search_term = ingredient_id.replace("-", " ")
                        if search_term in combined_text:
                            matched_ingredients.append(ingredient_id)

                    recipe_data = {
                        "id": recipe_id,
                        "title": page_title
                        if page_title
                        else recipe_id.replace("-", " ").title(),
                        "source": "khairul-aming",
                        "videoUrl": current_url,
                        "ingredients": matched_ingredients,
                        "prepTime": "45 min",
                        "servings": 4,
                    }

                    recipes_list.append(recipe_data)
                    seen_recipe_ids.add(recipe_id)
                    print(
                        f"-> Captured recipe text! Matched ingredients: {matched_ingredients}"
                    )

            for link in soup.find_all("a", href=True):
                full_url = urljoin(current_url, link["href"])
                if (
                    urlparse(full_url).netloc == domain
                    and full_url not in visited
                    and full_url not in to_visit
                ):
                    to_visit.append(full_url)

        except Exception as e:
            continue

    output_data = {"recipes": recipes_list}
    with open("scraped_recipes.json", "w", encoding="utf-8") as f:
        json.dump(output_data, f, indent=2, ensure_ascii=False)

    print(
        f"\nSuccess! Saved {len(recipes_list)} authentic recipes to scraped_recipes.json!"
    )


if __name__ == "__main__":
    crawl_khairul_aming_recipes("https://khairulaming.my")

--- Starting Deep Scan on khairulaming.my ---

Target Match Found! Processing page: https://khairulaming.my/resipi-ayam-penyet-sihat-dengan-air-fryer/
-> Captured recipe text! Matched ingredients: []

Target Match Found! Processing page: https://khairulaming.my/nasi-minyak-gulai-ayam-acar/
-> Captured recipe text! Matched ingredients: []

Success! Saved 2 authentic recipes to scraped_recipes.json!
